# 02-deit-train_fully_commented

Fully commented version. Comments explain the purpose of each important line/block and note Windows/Linux/macOS path considerations.


In [ ]:
# 02-deit-train_fully_commented: FULLY COMMENTED VERSION
# Every important code line/block includes a comment explaining what it does and why it is used.
# For Windows/Linux/macOS users, replace Kaggle-only paths such as /kaggle/working with your local project folder.



# ============================================================================
# # DEIT
# 
# ## Programmer Notes
# 
# This notebook now includes comments for Kaggle and local Windows/Linux/macOS use. The code remains Kaggle-compatible, but local users must replace Kaggle paths and secrets with local folders/environment variables.
# ============================================================================


## Cell 1
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# -----------------------------------------------------------------------------
# CROSS-PLATFORM NOTES FOR WINDOWS / LINUX / macOS
# -----------------------------------------------------------------------------
# This notebook was originally written for Kaggle. Kaggle uses Linux paths such
# as /kaggle/working and /kaggle/input. Those paths will NOT exist on Windows,
# macOS, or a normal Linux computer unless you create them manually.
#
# For local Windows/Linux/macOS use:
#   1. Create a project folder, for example:
#        Windows: C:\Users\YourName\CPE
#        macOS:   /Users/YourName/CPE
#        Linux:   /home/YourName/CPE
#   2. Replace Kaggle-only paths like /kaggle/working with your project folder.
#   3. Prefer os.path.join(...) or pathlib.Path(...) instead of typing slashes.
#      Python will automatically handle Windows backslashes and macOS/Linux
#      forward slashes.
#   4. GPU training needs an NVIDIA GPU + CUDA-compatible PyTorch. On Kaggle,
#      enable GPU in Notebook Settings. On local machines, install the correct
#      PyTorch build from the official PyTorch selector.
# -----------------------------------------------------------------------------
# CELL PURPOSE:
# Installs/imports libraries needed for DeiT image classification training.
# Kaggle users: enable GPU. Local users: verify torch.cuda.is_available().

# Purpose: Imports a Python module/library needed by the notebook.
import subprocess, sys, warnings
# Purpose: Hides non-critical warning messages so notebook output stays readable.
warnings.filterwarnings("ignore")
# Purpose: Loops through a list/collection and repeats the indented code for each item.
for pkg in ["transformers", "datasets", "accelerate", "matplotlib"]:
    # Purpose: Starts a safe block for code that might fail on some systems.
    try:
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        __import__(pkg)
    # Purpose: Handles an expected error so the notebook can continue or show a clearer message.
    except ImportError:
        # Purpose: Runs a pip command from inside Python to install missing packages automatically.
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
# Purpose: Imports a Python module/library needed by the notebook.
import torch
# Purpose: Stops the notebook when no CUDA GPU is detected, because training will be too slow or unsupported.
if not torch.cuda.is_available():
    # Purpose: Stops execution with a clear error when a required condition is missing.
    raise RuntimeError
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("CUDA OK", torch.__version__)


## Cell 2
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Loads config.json, resolves dataset folders, sets random seeds, and prepares
# logging. Local users should update KAGGLE_INPUT_ROOT/WORKING_CONFIG or place
# config.json in an accessible local folder.

# Purpose: Imports a Python module/library needed by the notebook.
import json, os, csv, random
# Purpose: Imports specific classes/functions so the code can use them directly.
from datetime import datetime
# Purpose: Imports a Python module/library needed by the notebook.
import numpy as np
# Purpose: Imports a Python module/library needed by the notebook.
import matplotlib.pyplot as plt
# Purpose: Imports a Python module/library needed by the notebook.
import torch
# Purpose: Imports a Python module/library needed by the notebook.
import torch.nn.functional as F
# Purpose: Imports specific classes/functions so the code can use them directly.
from PIL import Image
# Purpose: Imports specific classes/functions so the code can use them directly.
from torchvision import transforms
# Purpose: Imports specific classes/functions so the code can use them directly.
from transformers import (DeiTForImageClassificationWithTeacher,DeiTImageProcessor,TrainingArguments,Trainer,EarlyStoppingCallback,)
# Purpose: Imports specific classes/functions so the code can use them directly.
from datasets import Dataset as HFDataset
# Purpose: Imports specific classes/functions so the code can use them directly.
from sklearn.model_selection import StratifiedKFold
# Purpose: Imports specific classes/functions so the code can use them directly.
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,classification_report, confusion_matrix,)

# Purpose: Imports a Python module/library needed by the notebook.
import os

# Purpose: Creates or updates a variable used later in the notebook workflow.
KAGGLE_INPUT_ROOT = "/kaggle/input/datasets/nikachuu/data-prep"
# Purpose: Creates or updates a variable used later in the notebook workflow.
WORKING_CONFIG = "/kaggle/working/config.json"
# Purpose: Creates or updates a variable used later in the notebook workflow.
INPUT_CONFIG = os.path.join(KAGGLE_INPUT_ROOT, "config.json")

# Purpose: Defines a reusable function so the same logic can be called multiple times.
def load_config():
    # Purpose: Runs the indented code only when the condition is true.
    if os.path.isfile(WORKING_CONFIG):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config_path = WORKING_CONFIG
    # Purpose: Checks another condition if the previous if condition was false.
    elif os.path.isfile(INPUT_CONFIG):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config_path = INPUT_CONFIG
    # Purpose: Runs this block when none of the previous conditions matched.
    else:
        # Purpose: Stops execution with a clear error when a required condition is missing.
        raise FileNotFoundError
    # Purpose: Opens a file safely and automatically closes it after the block finishes.
    with open(config_path) as f:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config = json.load(f)
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    roots = [config.get("dataset_path"),os.path.join(KAGGLE_INPUT_ROOT, "dataset"),"/kaggle/working/dataset",]
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for root in roots:
        # Purpose: Runs the indented code only when the condition is true.
        if not root:
            # Purpose: Skips the rest of the current loop iteration and moves to the next item.
            continue
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        train_ann = os.path.join(root, "train", "_annotations.coco.json")
        # Purpose: Runs the indented code only when the condition is true.
        if os.path.isfile(train_ann):
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["dataset_path"] = root
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["train_ann"] = train_ann
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["valid_ann"] = os.path.join(root, "valid", "_annotations.coco.json")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["test_ann"] = os.path.join(root, "test", "_annotations.coco.json")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["train_img_dir"] = os.path.join(root, "train")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["valid_img_dir"] = os.path.join(root, "valid")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["test_img_dir"] = os.path.join(root, "test")
            # Purpose: Exits the current loop early because the needed condition was already found.
            break
    # Purpose: Runs this block when none of the previous conditions matched.
    else:
        # Purpose: Stops execution with a clear error when a required condition is missing.
        raise FileNotFoundError
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    config["save_dir"] = config.get("save_dir") or "/kaggle/working/results"
    # Purpose: Creates the output folder if it does not already exist.
    os.makedirs(config["save_dir"], exist_ok=True)
    # Purpose: Sends the computed result back to the function caller.
    return config, config_path

# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
config, CONFIG_PATH = load_config()

# Purpose: Sets the experiment name used in saved folders, plots, and results logs.
Experiment_Name = config["Experiment_Name"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
deit_variant = config["deit_variant"]
# Purpose: Selects which Roboflow dataset version to download/use.
dataset_version = config["dataset_version"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
Confidence_Threshold = config["Confidence_Threshold"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
epochs_deit = config["epochs_deit"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
batch_size = config["batch_size"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
lr_deit = config["lr_deit"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
weight_decay = config["weight_decay"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
PATIENCE = config["PATIENCE"]
# Purpose: Sets the fixed seed value used for reproducible splits and training behavior.
SEED = config["SEED"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
BASELINE_ACC = config["BASELINE_ACC"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
save_dir = config["save_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
train_ann = config["train_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
valid_ann = config["valid_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
test_ann = config["test_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
train_img_dir = config["train_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
valid_img_dir = config["valid_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
test_img_dir = config["test_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
log_file = os.path.join(save_dir, "results.csv")

# Purpose: Fixes Python random behavior so experiments are more reproducible.
random.seed(SEED)
# Purpose: Fixes NumPy random behavior so data splits and metrics are reproducible.
np.random.seed(SEED)
# Purpose: Fixes PyTorch random behavior for more reproducible training.
torch.manual_seed(SEED)
# Purpose: Fixes PyTorch CUDA random behavior when using one or more GPUs.
torch.cuda.manual_seed_all(SEED)
# Purpose: Chooses GPU when available, otherwise falls back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Purpose: Creates the output folder if it does not already exist.
os.makedirs(save_dir, exist_ok=True)

# Purpose: Defines a reusable function so the same logic can be called multiple times.
def log_result(exp, model, acc, prec, rec, f1, thresh, variant, ds_ver, notes=""):
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"),"experiment": exp, "model": model,"accuracy": f"{acc:.2f}", "precision": f"{prec:.2f}","recall": f"{rec:.2f}", "f1": f"{f1:.2f}","threshold": thresh, "variant": variant,"dataset_version": ds_ver, "notes": notes,}
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    write_header = not os.path.exists(log_file)
    # Purpose: Opens a file safely and automatically closes it after the block finishes.
    with open(log_file, "a", newline="") as f:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        w = csv.DictWriter(f, fieldnames=row.keys())
        # Purpose: Runs the indented code only when the condition is true.
        if write_header:
            # Purpose: Writes a row/header into the CSV results file.
            w.writeheader()
        # Purpose: Writes a row/header into the CSV results file.
        w.writerow(row)
    # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
    print("Logged to", log_file)

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Config", CONFIG_PATH)
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Dataset", config["dataset_path"])
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Results", save_dir)
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Device", DEVICE)


## Cell 3
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Loads COCO image samples, converts them into Healthy/Infected labels, and
# prepares image preprocessing/augmentation for DeiT.

# Purpose: Loads the DeiT image processor, which resizes/normalizes images for the model.
processor = DeiTImageProcessor.from_pretrained(deit_variant)

# Purpose: Defines a helper that converts COCO annotations into image classification samples.
def load_samples(ann_file, img_dir):
    # Purpose: Opens a file safely and automatically closes it after the block finishes.
    with open(ann_file) as f:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        coco = json.load(f)
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    amap = {a["image_id"]: a["category_id"] for a in coco["annotations"]}
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    samples = []
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for img in coco["images"]:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        cid = amap.get(img["id"])
        # Purpose: Runs the indented code only when the condition is true.
        if cid is None:
            # Purpose: Skips the rest of the current loop iteration and moves to the next item.
            continue
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        label = 1 if "infected" in cats.get(cid, "").lower() else 0
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        path = os.path.join(img_dir, img["file_name"])
        # Purpose: Runs the indented code only when the condition is true.
        if os.path.exists(path):
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            samples.append({"image_path": path, "label": label})
    # Purpose: Sends the computed result back to the function caller.
    return samples

# Purpose: Creates image augmentation steps to make DeiT more robust to visual variation.
domain_aug = transforms.Compose([
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    transforms.RandomHorizontalFlip(p=0.5),
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    transforms.RandomVerticalFlip(p=0.3),
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    transforms.RandomRotation(degrees=15),
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    transforms.RandomResizedCrop(size=224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),])

# Purpose: Defines preprocessing for training images, including augmentation.
def preprocess_train(s):
    # Purpose: Loads the image file and converts it to RGB.
    img = Image.open(s["image_path"]).convert("RGB")
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    img = domain_aug(img)
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    inp = processor(images=img, return_tensors="pt")
    # Purpose: Sends the computed result back to the function caller.
    return {"pixel_values": inp["pixel_values"].squeeze(0), "labels": int(s["label"])}

# Purpose: Defines preprocessing for validation/test images without random augmentation.
def preprocess_eval(s):
    # Purpose: Loads the image file and converts it to RGB.
    img = Image.open(s["image_path"]).convert("RGB")
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    inp = processor(images=img, return_tensors="pt")
    # Purpose: Sends the computed result back to the function caller.
    return {"pixel_values": inp["pixel_values"].squeeze(0), "labels": int(s["label"])}

# Purpose: Creates the Hugging Face Dataset for training samples.
train_hf = HFDataset.from_list(load_samples(train_ann, train_img_dir))
# Purpose: Creates the Hugging Face Dataset for validation samples.
valid_hf = HFDataset.from_list(load_samples(valid_ann, valid_img_dir))
# Purpose: Creates the Hugging Face Dataset for test samples.
test_hf = HFDataset.from_list(load_samples(test_ann, test_img_dir))
# Purpose: Creates the Hugging Face Dataset for training samples.
train_hf = train_hf.map(preprocess_train, remove_columns=train_hf.column_names)
# Purpose: Creates the Hugging Face Dataset for validation samples.
valid_hf = valid_hf.map(preprocess_eval, remove_columns=valid_hf.column_names)
# Purpose: Creates the Hugging Face Dataset for test samples.
test_hf = test_hf.map(preprocess_eval, remove_columns=test_hf.column_names)
# Purpose: Loops through a list/collection and repeats the indented code for each item.
for ds in [train_hf, valid_hf, test_hf]:
    # Purpose: Tells Hugging Face Datasets to return PyTorch tensors.
    ds.set_format("torch", columns=["pixel_values", "labels"])

# Purpose: Defines how multiple DeiT samples are stacked into one training batch.
def deit_collate_fn(batch):
    # Purpose: Sends the computed result back to the function caller.
    return {
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "labels": torch.tensor([b["labels"] for b in batch], dtype=torch.long),}

# Purpose: Defines a fallback function to extract labels from a dataset.
def labels_from_dataset(ds):
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    out = []
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for i in range(len(ds)):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        y = ds[i]["labels"]
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        out.append(y.item() if torch.is_tensor(y) else int(y))
    # Purpose: Sends the computed result back to the function caller.
    return np.array(out)

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("train", len(train_hf), "valid", len(valid_hf), "test", len(test_hf))
# Purpose: Checks an assumption and stops the notebook if the assumption is false.
assert len(valid_hf) > 0


## Cell 4
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Builds the DeiT model, defines evaluation metrics, defines a custom trainer
# for the distilled DeiT output, and starts training.

# Purpose: Loads the pretrained DeiT model and changes it for two classes: Healthy/Infected.
deit_model = DeiTForImageClassificationWithTeacher.from_pretrained(
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    deit_variant, num_labels=2,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    id2label={0: "Healthy", 1: "Infected"},
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    label2id={"Healthy": 0, "Infected": 1},
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    ignore_mismatched_sizes=True,).to(DEVICE)

# Purpose: Defines the metrics shown during evaluation.
def compute_metrics(eval_pred):
    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
    logits, labels = eval_pred
    # Purpose: Runs the indented code only when the condition is true.
    if isinstance(logits, tuple):
        # Purpose: Stores raw model prediction scores before converting them to class labels.
        logits = (logits[0] + logits[1]) / 2
    # Purpose: Converts model scores into predicted class IDs.
    preds = np.argmax(logits, axis=-1)
    # Purpose: Sends the computed result back to the function caller.
    return {
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "accuracy": accuracy_score(labels, preds),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),}

# Purpose: Creates a custom Trainer so both DeiT classifier and distillation heads contribute to loss.
class DeiTDistilledTrainer(Trainer):
    # Purpose: Defines a reusable function so the same logic can be called multiple times.
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        labels = inputs.pop("labels", None)
        # Purpose: Runs the indented code only when the condition is true.
        if labels is None:
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            labels = inputs.pop("label", None)
        # Purpose: Runs the indented code only when the condition is true.
        if labels is None:
            # Purpose: Stops execution with a clear error when a required condition is missing.
            raise KeyError("No labels in batch. Re-run dataset cell.")
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        outputs = model(**inputs)
        # Purpose: Computes the average classification loss from the two DeiT output heads.
        loss = 0.5 * F.cross_entropy(outputs.cls_logits, labels) +                0.5 * F.cross_entropy(outputs.distillation_logits, labels)
        # Purpose: Sends the computed result back to the function caller.
        return (loss, outputs) if return_outputs else loss

# Purpose: Defines all Hugging Face Trainer settings such as epochs, batch size, saving, and mixed precision.
training_args = TrainingArguments(
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    output_dir=os.path.join(save_dir, "deit_" + Experiment_Name),
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    num_train_epochs=epochs_deit,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    per_device_train_batch_size=batch_size,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    per_device_eval_batch_size=batch_size,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    learning_rate=lr_deit,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    weight_decay=weight_decay,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    max_grad_norm=1.0,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    lr_scheduler_type="cosine",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    warmup_ratio=0.10,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    eval_strategy="epoch",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    save_strategy="epoch",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    load_best_model_at_end=True,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    metric_for_best_model="accuracy",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    greater_is_better=True,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    logging_strategy="epoch",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    report_to="none",
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    fp16=torch.cuda.is_available(),
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    seed=SEED,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    data_seed=SEED,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    save_total_limit=2,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    dataloader_num_workers=0,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    remove_unused_columns=False,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    label_names=["labels"],)

# Purpose: Creates the Trainer object that will handle training and validation.
trainer = DeiTDistilledTrainer(
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    model=deit_model,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    args=training_args,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    train_dataset=train_hf,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    eval_dataset=valid_hf,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    data_collator=deit_collate_fn,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    compute_metrics=compute_metrics,
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],)

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Training DeiT...")
# Purpose: Starts DeiT training and stores the training summary.
train_result = trainer.train()
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Training loss:", round(train_result.training_loss, 4))


## Cell 5
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Evaluates DeiT on the test set, prints metrics, saves the confusion matrix,
# and appends results to results.csv.

# Purpose: Runs the trained DeiT model on the test set.
test_output = trainer.predict(test_hf)
# Purpose: Stores raw model prediction scores before converting them to class labels.
logits = test_output.predictions
# Purpose: Creates or updates a variable used later in the notebook workflow.
labels = test_output.label_ids
# Purpose: Runs the indented code only when the condition is true.
if labels is None:
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    labels = labels_from_dataset(test_hf)
# Purpose: Runs the indented code only when the condition is true.
if isinstance(logits, tuple):
    # Purpose: Stores raw model prediction scores before converting them to class labels.
    logits = (logits[0] + logits[1]) / 2
# Purpose: Converts model scores into predicted class IDs.
preds = np.argmax(logits, axis=-1)

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print(classification_report(labels, preds, target_names=["Healthy", "Infected"], digits=4))

# Purpose: Computes the confusion matrix for Healthy vs Infected predictions.
cm = confusion_matrix(labels, preds)
# Purpose: Creates a Matplotlib figure and axes used for drawing a chart.
fig, ax = plt.subplots(figsize=(6, 5))
# Purpose: Configures one part of the chart for clearer visualization.
ax.imshow(cm, cmap="Reds")
# Purpose: Loops through a list/collection and repeats the indented code for each item.
for i in range(2):
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for j in range(2):
        # Purpose: Configures one part of the chart for clearer visualization.
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
# Purpose: Configures one part of the chart for clearer visualization.
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
# Purpose: Configures one part of the chart for clearer visualization.
ax.set_xticklabels(["Healthy", "Infected"])
# Purpose: Configures one part of the chart for clearer visualization.
ax.set_yticklabels(["Healthy", "Infected"])
# Purpose: Configures one part of the chart for clearer visualization.
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
# Purpose: Configures one part of the chart for clearer visualization.
ax.set_title("DeiT confusion matrix " + Experiment_Name)
# Purpose: Automatically adjusts spacing so chart labels do not overlap.
plt.tight_layout()
# Purpose: Saves the generated chart as an image file for reports/thesis documentation.
plt.savefig(os.path.join(save_dir, "deit_cm_" + Experiment_Name + ".png"), dpi=150)
# Purpose: Displays the chart inside the notebook output.
plt.show()

# Purpose: Creates or updates a variable used later in the notebook workflow.
deit_acc = accuracy_score(labels, preds) * 100
# Purpose: Creates or updates a variable used later in the notebook workflow.
deit_prec = precision_score(labels, preds, average="weighted", zero_division=0) * 100
# Purpose: Creates or updates a variable used later in the notebook workflow.
deit_rec = recall_score(labels, preds, average="weighted", zero_division=0) * 100
# Purpose: Creates or updates a variable used later in the notebook workflow.
deit_f1 = f1_score(labels, preds, average="weighted", zero_division=0) * 100

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Accuracy:", round(deit_acc, 2), "Baseline:", BASELINE_ACC, "Diff:", round(deit_acc - BASELINE_ACC, 2))

# Purpose: Sets the folder path where the best DeiT model will be saved.
DEIT_SAVE = os.path.join(save_dir, "deit_best_model")
# Purpose: Saves the trained DeiT model for later inference or deployment.
trainer.save_model(DEIT_SAVE)
# Purpose: Creates or updates a variable used later in the notebook workflow.
config["DEIT_SAVE"] = DEIT_SAVE
# Purpose: Opens a file safely and automatically closes it after the block finishes.
with open(WORKING_CONFIG, "w") as f:
    # Purpose: Writes Python dictionary/list data into a JSON file.
    json.dump(config, f, indent=2)
# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
log_result(Experiment_Name, "DeiT-Small-Distilled", deit_acc, deit_prec, deit_rec, deit_f1,
           # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
           Confidence_Threshold, deit_variant, dataset_version)
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Saved", DEIT_SAVE)


## Cell 6
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Plots training history: losses, validation accuracy, and overfitting gap.
# The output image is saved inside save_dir.

# Purpose: Reads the Trainer history containing losses and validation accuracy.
logs = trainer.state.log_history
# Purpose: Extracts training loss values from the training log.
tr_losses = [l["loss"] for l in logs if "loss" in l and "eval_loss" not in l]
# Purpose: Extracts validation loss values from the training log.
val_losses = [l["eval_loss"] for l in logs if "eval_loss" in l]
# Purpose: Extracts validation accuracy values from the training log.
val_accs = [l["eval_accuracy"] * 100 for l in logs if "eval_accuracy" in l]

# Purpose: Creates or updates a variable used later in the notebook workflow.
C_VIOLET = "#6F42C1"   
# Purpose: Creates or updates a variable used later in the notebook workflow.
C_MAROON = "#800000"     
# Purpose: Creates or updates a variable used later in the notebook workflow.
C_BLUE = "#007BFF"       
# Purpose: Creates or updates a variable used later in the notebook workflow.
C_BROWN = "#8B4513"     
# Purpose: Creates or updates a variable used later in the notebook workflow.
C_ORANGE = "#FF7F0E"      

# Purpose: Creates a Matplotlib figure and axes used for drawing a chart.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Purpose: Configures one part of the chart for clearer visualization.
axes[0].plot(tr_losses, label="Train", color=C_VIOLET, linewidth=2, marker='o', markersize=4)
# Purpose: Configures one part of the chart for clearer visualization.
axes[0].plot(val_losses, label="Val", color=C_MAROON, linewidth=2, marker='s', markersize=4)
# Purpose: Configures one part of the chart for clearer visualization.
axes[0].set_title("Loss", fontsize=12, fontweight="bold", pad=10)
# Purpose: Configures one part of the chart for clearer visualization.
axes[0].grid(True, linestyle="--", alpha=0.3)
# Purpose: Configures one part of the chart for clearer visualization.
axes[0].legend(frameon=True, facecolor="#F8F9FA", edgecolor="none")

# Purpose: Configures one part of the chart for clearer visualization.
axes[1].plot(val_accs, color=C_BLUE, linewidth=2.5, marker='v', markersize=5, label="Val Acc")
# Purpose: Configures one part of the chart for clearer visualization.
axes[1].axhline(BASELINE_ACC, linestyle="--", color=C_BROWN, linewidth=1.5, label="Baseline")
# Purpose: Configures one part of the chart for clearer visualization.
axes[1].set_title("Val Accuracy (%)", fontsize=12, fontweight="bold", pad=10)
# Purpose: Configures one part of the chart for clearer visualization.
axes[1].grid(True, linestyle="--", alpha=0.3)
# Purpose: Configures one part of the chart for clearer visualization.
axes[1].legend(frameon=True, facecolor="#F8F9FA", edgecolor="none")

# Purpose: Runs the indented code only when the condition is true.
if len(tr_losses) == len(val_losses):
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    gap = [v - t for v, t in zip(val_losses, tr_losses)]
    
    # Purpose: Configures one part of the chart for clearer visualization.
    axes[2].bar(range(len(gap)), gap, color=C_ORANGE, edgecolor="none", alpha=0.85, width=0.6)
    # Purpose: Configures one part of the chart for clearer visualization.
    axes[2].axhline(0, color="#CCCCCC", linewidth=1)
    # Purpose: Configures one part of the chart for clearer visualization.
    axes[2].set_title("Val Minus Train Loss", fontsize=12, fontweight="bold", pad=10)
    # Purpose: Configures one part of the chart for clearer visualization.
    axes[2].grid(True, linestyle="--", alpha=0.3)

# Purpose: Configures or displays the Matplotlib output.
plt.suptitle("DeiT Model Analysis: " + Experiment_Name, fontsize=14, fontweight="bold", y=1.02)
# Purpose: Automatically adjusts spacing so chart labels do not overlap.
plt.tight_layout()

# Purpose: Saves the generated chart as an image file for reports/thesis documentation.
plt.savefig(os.path.join(save_dir, "deit_history_" + Experiment_Name + ".png"), dpi=200, bbox_inches="tight")
# Purpose: Displays the chart inside the notebook output.
plt.show()


## Cell 7
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Optional 5-fold cross-validation. This is slower than normal training.
# Use it when you need stronger thesis/report evidence for model stability.

# Purpose: Defines optional k-fold cross-validation for stronger model stability evidence.
def cross_validate(n_splits=5):
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    samples = load_samples(train_ann, train_img_dir)
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    lbls = [s["label"] for s in samples]
    # Purpose: Creates stratified folds so each fold keeps a similar class distribution.
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    fold_accs = []
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for fold, (tr_i, val_i) in enumerate(skf.split(samples, lbls)):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        tr_ds = HFDataset.from_list([samples[i] for i in tr_i])
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        val_ds = HFDataset.from_list([samples[i] for i in val_i])
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        tr_ds = tr_ds.map(preprocess_train, remove_columns=tr_ds.column_names)
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        val_ds = val_ds.map(preprocess_eval, remove_columns=val_ds.column_names)
        # Purpose: Loops through a list/collection and repeats the indented code for each item.
        for ds in [tr_ds, val_ds]:
            # Purpose: Tells Hugging Face Datasets to return PyTorch tensors.
            ds.set_format("torch", columns=["pixel_values", "labels"])
        # Purpose: Starts a multi-line Python structure/call; the following indented lines provide its values.
        m = DeiTForImageClassificationWithTeacher.from_pretrained(
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            deit_variant, num_labels=2, ignore_mismatched_sizes=True)
        # Purpose: Starts a multi-line Python structure/call; the following indented lines provide its values.
        fa = TrainingArguments(
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            output_dir="/tmp/fold_" + str(fold), num_train_epochs=10,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            per_device_train_batch_size=batch_size,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            per_device_eval_batch_size=batch_size,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            learning_rate=lr_deit, weight_decay=weight_decay,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            eval_strategy="epoch", save_strategy="no", logging_strategy="no",
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            report_to="none", fp16=torch.cuda.is_available(), seed=SEED,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            remove_unused_columns=False, dataloader_num_workers=0,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            label_names=["labels"],) 
        # Purpose: Starts a multi-line Python structure/call; the following indented lines provide its values.
        ft = DeiTDistilledTrainer(
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            model=m, args=fa, train_dataset=tr_ds, eval_dataset=val_ds,
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            data_collator=deit_collate_fn, compute_metrics=compute_metrics)
        # Purpose: Calls a Trainer method to train, evaluate, save, or inspect the model.
        ft.train()
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        val_labels = labels_from_dataset(val_ds)
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        res = ft.predict(val_ds)
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        raw = res.predictions
        # Purpose: Runs the indented code only when the condition is true.
        if isinstance(raw, tuple):
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            raw = (raw[0] + raw[1]) / 2
        # Purpose: Converts model scores into predicted class IDs.
        preds = np.argmax(raw, axis=-1)
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        y_true = res.label_ids if res.label_ids is not None else val_labels
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        acc = accuracy_score(y_true, preds) * 100
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        fold_accs.append(acc)
        # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
        print("Fold", fold + 1, "accuracy", round(acc, 2))
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        del m, ft
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        torch.cuda.empty_cache()
    # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
    print("CV mean", round(np.mean(fold_accs), 2), "std", round(np.std(fold_accs), 2))
    # Purpose: Sends the computed result back to the function caller.
    return fold_accs

# Purpose: Creates or updates a variable used later in the notebook workflow.
cv_accs = cross_validate(5)
